# End-to-end pretrained PANNs CNN14 for ICBHI binary classification

This notebook replaces the original **CNN14 feature extractor + StandardScaler + PCA + SVM** pipeline with a trainable deep-learning classifier.

- Pretrained PANNs CNN14 initializes the audio backbone.
- A trainable two-class head predicts **Normal (0)** vs **Abnormal (1)**.
- CNN14 backbone parameters and the new classification head are optimized jointly.
- Validation is split at the patient level from the official training set.
- The official test set is not used for model selection.
- Best checkpoint is selected by validation HS-score; last checkpoint is saved separately.

**Dataset assumptions:** `split_metadata.csv` has `source_file`, `start`, `end`, `crackle`, `wheeze`, and `split` columns. `source_file` follows the ICBHI convention where patient ID is the part before the first underscore. `Cnn14_mAP=0.431.pth` is the pretrained PANNs checkpoint compatible with the repository's `Cnn14` class.

In [1]:
# 1. Imports and reproducibility
import os
import random
import json
import importlib.util
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import confusion_matrix, accuracy_score


D:\.JKU CS courses\Master Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#from audioset_tagging_cnn_master.pytorch.models import Cnn14
SRC_DIR = Path.cwd()
PYTORCH_DIR = SRC_DIR / "audioset_tagging_cnn_master" / "pytorch"
MODELS_PATH = PYTORCH_DIR / "models.py"
LOCAL_UTILS_PATH = PYTORCH_DIR / "pytorch_utils.py"

assert MODELS_PATH.is_file(), f"Missing file: {MODELS_PATH}"
assert LOCAL_UTILS_PATH.is_file(), f"Missing file: {LOCAL_UTILS_PATH}"

# Make the local helper available under the name expected by AudioSet.
utils_spec = importlib.util.spec_from_file_location(
    "pytorch_utils", LOCAL_UTILS_PATH
)
utils_module = importlib.util.module_from_spec(utils_spec)
sys.modules["pytorch_utils"] = utils_module


# Execute the local helper module
utils_spec.loader.exec_module(utils_module)

print("Local helper loaded:", utils_module.__file__)
print("Has forward:", hasattr(utils_module, "forward"))

# Load models.py directly, without executing the package __init__.py
models_spec = importlib.util.spec_from_file_location(
    "audioset_cnn14_models", MODELS_PATH
)
models_module = importlib.util.module_from_spec(models_spec)
sys.modules[models_spec.name] = models_module
models_spec.loader.exec_module(models_module)

Cnn14 = models_module.Cnn14
print("CNN14 imported successfully:", Cnn14)



SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Local helper loaded: D:\.JKU CS courses\Master Thesis\src\audioset_tagging_cnn_master\pytorch\pytorch_utils.py
Has forward: True
CNN14 imported successfully: <class 'audioset_cnn14_models.Cnn14'>
Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [3]:
# 2. Paths and hyperparameters — update paths for your local project
AUDIO_DIR = Path("../datasets/raw/ICBHI_final_database/audios")
METADATA_PATH = Path("../datasets/split_metadata.csv")
PRETRAINED_CHECKPOINT = Path("Cnn14_mAP=0.431.pth")
OUTPUT_DIR = Path("../models/cnn14_end_to_end_binary")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16_000
SEGMENT_SECONDS = 10.0  # fixed input length for batching; segments are cropped using metadata start/end
BATCH_SIZE = 8
NUM_WORKERS = 0  # safe default on Windows/Jupyter
EPOCHS = 20
VAL_SIZE = 0.20
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
USE_AMP = False #(device.type == "cuda")

BEST_CHECKPOINT = OUTPUT_DIR / "cnn14_best_val_hs.pt"
LAST_CHECKPOINT = OUTPUT_DIR / "cnn14_last_epoch.pt"
RESULTS_PATH = OUTPUT_DIR / "test_results.json"

print("Audio directory:", AUDIO_DIR.resolve())
print("Metadata:", METADATA_PATH.resolve())
print("Pretrained checkpoint:", PRETRAINED_CHECKPOINT.resolve())

Audio directory: D:\.JKU CS courses\Master Thesis\datasets\raw\ICBHI_final_database\audios
Metadata: D:\.JKU CS courses\Master Thesis\datasets\split_metadata.csv
Pretrained checkpoint: D:\.JKU CS courses\Master Thesis\src\Cnn14_mAP=0.431.pth


In [4]:
# 3. Load and validate metadata
for required_path in (AUDIO_DIR, METADATA_PATH, PRETRAINED_CHECKPOINT):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path does not exist: {required_path.resolve()}\n"
            "Update AUDIO_DIR, METADATA_PATH, and PRETRAINED_CHECKPOINT to match your local files."
        )

df = pd.read_csv(METADATA_PATH)
required_columns = {"source_file", "start", "end", "crackle", "wheeze", "split"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Metadata is missing required columns: {sorted(missing)}")

df = df.copy()
df["source_file"] = df["source_file"].astype(str).str.strip()
df["split"] = df["split"].astype(str).str.strip().str.lower()
split_map = {"train": "train", "training": "train", "test": "test", "testing": "test"}
unknown = sorted(set(df["split"]) - set(split_map))
if unknown:
    raise ValueError(f"Unexpected split values {unknown}; expected official train/test labels.")
df["split"] = df["split"].map(split_map)

def patient_id_from_source(source_file):
    stem = Path(source_file).stem
    if "_" not in stem:
        raise ValueError(
            f"Cannot infer patient ID from '{source_file}'. Expected ICBHI recording filename "
            "with patient ID before the first underscore."
        )
    return stem.split("_", 1)[0]

df["patient_id"] = df["source_file"].map(patient_id_from_source)
if df[list(required_columns)].isna().any().any():
    raise ValueError("Metadata contains missing values in required columns.")

official_train = df[df["split"] == "train"].copy()
official_test = df[df["split"] == "test"].copy()
if official_train.empty or official_test.empty:
    raise ValueError("Both official train and official test rows must be present.")

""" 
since there is data overlap in the official data split, the error << raise ValueError() >>
will be ignored and replaced with a warning message << warnings.warn() >>
"""
overlap = set(official_train.patient_id) & set(official_test.patient_id)
if overlap:
    warnings.warn(
        f"\n!!! WARNING DATA LEAKAGE !!!\nPatient leakage in metadata: {len(overlap)} patient(s) appear in "
        f"both official train and test, e.g. {sorted(overlap)[:10]}. Correct split_metadata.csv before training!"
    )

print(f"Official train: {len(official_train):,} segments / {official_train.patient_id.nunique()} patients")
print(f"Official test:  {len(official_test):,} segments / {official_test.patient_id.nunique()} patients")

Official train: 4,142 segments / 79 patients
Official test:  2,756 segments / 49 patients


C:\Users\arebr\AppData\Local\Temp\ipykernel_32248\2842253417.py:48: UserWarning: 
!!! WARNING DATA LEAKAGE !!!
Patient leakage in metadata: 2 patient(s) appear in both official train and test, e.g. ['156', '218']. Correct split_metadata.csv before training!
  warnings.warn(


In [5]:
# 4. Patient-level validation split from official training data only

splitter = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)

tr_idx, va_idx = next(splitter.split(
                                    X=np.zeros(len(official_train)),
                                    y=official_train["label"],
                                    groups=official_train["patient_id"]
                                    )
)

train_df = official_train.iloc[tr_idx].reset_index(drop=True)
val_df = official_train.iloc[va_idx].reset_index(drop=True)
test_df = official_test.reset_index(drop=True)  # Keep official test untouched

# Verify patient separation
train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])

assert train_patients.isdisjoint(val_patients), "Train/validation leakage!"

# Binary labels: 0 = Normal, 1 = Abnormal
def binary_label(crackle, wheeze):
    return 0 if int(crackle) == 0 and int(wheeze) == 0 else 1

for frame in (train_df, val_df, test_df):
    frame["label"] = [
        binary_label(c, w)
        for c, w in zip(frame["crackle"], frame["wheeze"])
    ]

# Split sizes and patient counts
display(pd.DataFrame([
    {
        "split": "train",
        "segments": len(train_df),
        "patients": train_df["patient_id"].nunique()
    },
    {
        "split": "validation",
        "segments": len(val_df),
        "patients": val_df["patient_id"].nunique()
    },
    {
        "split": "official_test",
        "segments": len(test_df),
        "patients": test_df["patient_id"].nunique()
    }
]))

# Compare class distributions
for name, frame in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Official test", test_df)
]:
    print(f"\n{name} class counts:")
    print(frame["label"].value_counts().reindex([0, 1], fill_value=0))
    print(f"{name} class proportions:")
    print(frame["label"].value_counts(normalize=True).reindex(
        [0, 1], fill_value=0
    ).round(4))

print("\nNo patient overlap between train and validation.")


,split,segments,patients
0,train,3485,61
1,validation,657,18
2,official_test,2756,49



Train class counts:
label
0    1788
1    1697
Name: count, dtype: int64
Train class proportions:
label
0    0.5131
1    0.4869
Name: proportion, dtype: float64

Validation class counts:
label
0    275
1    382
Name: count, dtype: int64
Validation class proportions:
label
0    0.4186
1    0.5814
Name: proportion, dtype: float64

Official test class counts:
label
0    1579
1    1177
Name: count, dtype: int64
Official test class proportions:
label
0    0.5729
1    0.4271
Name: proportion, dtype: float64

No patient overlap between train and validation.


In [6]:
# 5. Audio dataset: crop each annotated segment, resample to 16 kHz, pad/trim to fixed duration
class ICBHIAudioDataset(Dataset):
    def __init__(self, frame, audio_dir):
        self.frame = frame.reset_index(drop=True).copy()
        self.audio_dir = Path(audio_dir)
        self.cache = {}

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        filename = row["source_file"]
        if not filename.lower().endswith(".wav"):
            filename += ".wav"
        wav_path = self.audio_dir / filename
        if not wav_path.exists():
            raise FileNotFoundError(f"Audio file not found: {wav_path}")

        if filename not in self.cache:
            audio, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)
            self.cache[filename] = audio
        audio = self.cache[filename]

        start = max(0, int(float(row["start"]) * SAMPLE_RATE))
        end = min(len(audio), int(float(row["end"]) * SAMPLE_RATE))
        segment = audio[start:end]
        if len(segment) == 0:
            raise ValueError(f"Empty segment for {filename}: start={row['start']}, end={row['end']}")

        target_len = int(SAMPLE_RATE * SEGMENT_SECONDS)
        if len(segment) < target_len:
            segment = np.pad(segment, (0, target_len - len(segment)))
        else:
            segment = segment[:target_len]

        # Return waveform [time] and integer class label.
        return torch.tensor(segment, dtype=torch.float32), torch.tensor(int(row["label"]), dtype=torch.long)

train_ds = ICBHIAudioDataset(train_df, AUDIO_DIR)
val_ds = ICBHIAudioDataset(val_df, AUDIO_DIR)
test_ds = ICBHIAudioDataset(test_df, AUDIO_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=(device.type == "cuda"))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=(device.type == "cuda"))

wave_batch, label_batch = next(iter(train_loader))
print("Waveform batch:", tuple(wave_batch.shape), "label batch:", tuple(label_batch.shape))

Waveform batch: (8, 160000) label batch: (8,)


In [16]:
# 6. Initialize pretrained CNN14 backbone and add a trainable binary classification head
# Load the original 527-class AudioSet model first so pretrained checkpoint dimensions match.
backbone = Cnn14(
    sample_rate=16000,
    window_size=1024,
    hop_size=320,
    mel_bins=64,
    fmin=50,
    fmax=8000,
    classes_num=527,
)

checkpoint = torch.load(PRETRAINED_CHECKPOINT, map_location="cpu", weights_only=False)
state_dict = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint
missing, unexpected = backbone.load_state_dict(state_dict, strict=False)
print("Pretrained checkpoint loaded.")
print("Missing keys:", missing[:10], "..." if len(missing) > 10 else "")
print("Unexpected keys:", unexpected[:10], "..." if len(unexpected) > 10 else "")

class CNN14BinaryClassifier(nn.Module):
    def __init__(self, cnn14_backbone, embedding_dim=2048, num_classes=2):
        super().__init__()
        self.backbone = cnn14_backbone
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, waveform):
        outputs = self.backbone(waveform, None)
        embedding = outputs["embedding"]
        logits = self.classifier(embedding)
        return logits

model = CNN14BinaryClassifier(backbone).to(device)


# Freeze the pretrained CNN14 backbone initially.
for parameter in model.backbone.parameters():
    parameter.requires_grad = False

# Train only the new binary classification head.
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

print(
    f"Trainable parameters: "
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

Pretrained checkpoint loaded.
Missing keys: [] 
Unexpected keys: [] 
Trainable parameters: 4,098


In [17]:
# 7. Metrics and epoch runner
def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    hs = (2 * sensitivity * specificity / (sensitivity + specificity)
          if (sensitivity + specificity) else 0.0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "HS_score": float(hs),
        "confusion_matrix": cm.tolist(),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def run_epoch(model, loader, training):
    model.train(training)
    if training:
        for module in model.backbone.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
                
    total_loss = 0.0
    y_true, y_pred = [], []

    for waveforms, labels in tqdm(loader, leave=False):
        waveforms = waveforms.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type,
                                dtype=torch.float16,
                                enabled=(USE_AMP and device.type == "cuda")
                               ):
                logits = model(waveforms)
            
                # Check model outputs before calculating the loss
                if not torch.isfinite(logits).all():
                    raise RuntimeError(f"Non-finite logits detected. Training={training}")
            
                loss = criterion(logits, labels)
            
            # Check loss BEFORE backward() or optimizer.step()
            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss detected. "
                    f"Training={training}, loss={loss.item()}"
                )


            if training:
                if USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()


        total_loss += loss.item() * waveforms.size(0)
        predictions = logits.argmax(dim=1)
        y_true.extend(labels.detach().cpu().tolist())
        y_pred.extend(predictions.detach().cpu().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(total_loss / len(loader.dataset))
    return metrics

In [18]:
# Test the freshly restored model in full precision
model.eval()

waveforms, labels = next(iter(train_loader))
waveforms = waveforms.to(device)

with torch.no_grad():
    logits = model(waveforms)

print("Waveforms finite:", torch.isfinite(waveforms).all().item())
print("Logits shape:", tuple(logits.shape))
print("Logits finite:", torch.isfinite(logits).all().item())

bad_buffers = [
    name for name, buf in model.named_buffers()
    if torch.is_floating_point(buf)
    and not torch.isfinite(buf).all()
]
print("Invalid model buffers:", bad_buffers)

if not torch.isfinite(logits).all():
    raise RuntimeError(
        "Fresh model still produces NaN/Inf logits. Do not start training."
    )

print("Full-precision forward pass passed.")


Waveforms finite: True
Logits shape: (8, 2)
Logits finite: True
Invalid model buffers: []
Full-precision forward pass passed.


In [23]:
batch = next(iter(train_loader))
waveforms, labels = batch[:2]

print("Waveform shape:", waveforms.shape)
print("Waveforms finite:", torch.isfinite(waveforms).all().item())
print("Waveform min/max:", waveforms.min().item(), waveforms.max().item())
print("Labels:", torch.unique(labels))
print("Labels finite:", torch.isfinite(labels).all().item())

Waveform shape: torch.Size([8, 160000])
Waveforms finite: True
Waveform min/max: -1.0015783309936523 1.0019997358322144
Labels: tensor([0, 1])
Labels finite: True


In [19]:
# 8. Train and save best/last checkpoints
history = []
best_val_hs = -1.0
best_epoch = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, training=True)
    val_metrics = run_epoch(model, val_loader, training=False)
    scheduler.step(val_metrics["HS_score"])
    lr_now = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch,
        "lr": lr_now,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_sensitivity": train_metrics["sensitivity"],
        "train_specificity": train_metrics["specificity"],
        "train_HS_score": train_metrics["HS_score"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_sensitivity": val_metrics["sensitivity"],
        "val_specificity": val_metrics["specificity"],
        "val_HS_score": val_metrics["HS_score"],
    })

    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_metrics": val_metrics,
        "seed": SEED,
        "class_mapping": {"Normal": 0, "Abnormal": 1},
        "config": {
            "sample_rate": SAMPLE_RATE,
            "segment_seconds": SEGMENT_SECONDS,
            "learning_rate": LEARNING_RATE,
            "batch_size": BATCH_SIZE,
        },
    }
    torch.save(payload, LAST_CHECKPOINT)

    if val_metrics["HS_score"] > best_val_hs:
        best_val_hs = val_metrics["HS_score"]
        best_epoch = epoch
        torch.save(payload, BEST_CHECKPOINT)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss={train_metrics['loss']:.4f}, acc={train_metrics['accuracy']:.4f}, "
        f"HS={train_metrics['HS_score']:.4f} | "
        f"val loss={val_metrics['loss']:.4f}, acc={val_metrics['accuracy']:.4f}, "
        f"SE={val_metrics['sensitivity']:.4f}, SP={val_metrics['specificity']:.4f}, "
        f"HS={val_metrics['HS_score']:.4f} | lr={lr_now:.2e}"
    )

history_df = pd.DataFrame(history)
display(history_df)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print("Best epoch:", best_epoch, "| best validation HS:", round(best_val_hs, 4))
print("Best checkpoint:", BEST_CHECKPOINT.resolve())
print("Last checkpoint:", LAST_CHECKPOINT.resolve())

Epoch 01/20 | train loss=0.6947, acc=0.5544, HS=0.5522 | val loss=0.6527, acc=0.6286, SE=0.5524, SP=0.7345, HS=0.6306 | lr=1.00e-03


Epoch 02/20 | train loss=0.6896, acc=0.5736, HS=0.5714 | val loss=0.6575, acc=0.5967, SE=0.3743, SP=0.9055, HS=0.5297 | lr=1.00e-03


Epoch 03/20 | train loss=0.6864, acc=0.5679, HS=0.5657 | val loss=0.6256, acc=0.6773, SE=0.8560, SP=0.4291, HS=0.5716 | lr=1.00e-03


Epoch 04/20 | train loss=0.6796, acc=0.5902, HS=0.5880 | val loss=0.6211, acc=0.6377, SE=0.9529, SP=0.2000, HS=0.3306 | lr=1.00e-03


Epoch 05/20 | train loss=0.6693, acc=0.5943, HS=0.5933 | val loss=0.6129, acc=0.6667, SE=0.8534, SP=0.4073, HS=0.5514 | lr=5.00e-04


...

Epoch 18/20 | train loss=0.6594, acc=0.6017, HS=0.6014 | val loss=0.6397, acc=0.6347, SE=0.4215, SP=0.9309, HS=0.5802 | lr=1.25e-04


Epoch 19/20 | train loss=0.6521, acc=0.6112, HS=0.6072 | val loss=0.6079, acc=0.6956, SE=0.8691, SP=0.4545, HS=0.5969 | lr=1.25e-04


Epoch 20/20 | train loss=0.6521, acc=0.6121, HS=0.6115 | val loss=0.6161, acc=0.6910, SE=0.7251, SP=0.6436, HS=0.6820 | lr=1.25e-04


,epoch,lr,train_loss,train_accuracy,train_sensitivity,train_specificity,train_HS_score,val_loss,val_accuracy,val_sensitivity,val_specificity,val_HS_score
0,1,0.001000,0.694718,0.554376,0.525633,0.581655,0.552227,0.652710,0.628615,0.552356,0.734545,0.630554
1,2,0.001000,0.689592,0.573601,0.543901,0.601790,0.571383,0.657463,0.596651,0.374346,0.905455,0.529697
2,3,0.001000,0.686389,0.567862,0.538598,0.595638,0.565683,0.625641,0.677321,0.856021,0.429091,0.571640
3,4,0.001000,0.679600,0.590244,0.559811,0.619128,0.587977,0.621060,0.637747,0.952880,0.200000,0.330609
4,5,0.000500,0.669287,0.594261,0.576311,0.611298,0.593289,0.612871,0.666667,0.853403,0.407273,0.551399
5,6,0.000500,0.680787,0.578766,0.550972,0.605145,0.576790,0.617509,0.687976,0.861257,0.447273,0.588778
6,7,0.000500,0.669368,0.588235,0.567472,0.607942,0.587010,0.625577,0.678843,0.698953,0.650909,0.674076
7,8,0.000500,0.665570,0.597704,0.575133,0.619128,0.596320,0.610474,0.697108,0.845550,0.490909,0.621176
8,9,0.000500,0.670953,0.589383,0.565115,0.612416,0.587815,0.611506,0.668189,0.874346,0.381818,0.531525
9,10,0.000500,0.665264,0.597131,0.565704,0.626957,0.594758,0.608938,0.640791,0.934555,0.232727,0.372654


Best epoch: 14 | best validation HS: 0.6866
Best checkpoint: D:\.JKU CS courses\Master Thesis\models\cnn14_end_to_end_binary\cnn14_best_val_hs.pt
Last checkpoint: D:\.JKU CS courses\Master Thesis\models\cnn14_end_to_end_binary\cnn14_last_epoch.pt


In [20]:
# 9. Final official test evaluation (run only after training/model selection is complete)
# Do not use official test metrics to tune hyperparameters.
def evaluate_checkpoint(checkpoint_path, loader, label):
    payload = torch.load(checkpoint_path, map_location=device, weights_only=False)
    eval_backbone = Cnn14(
        sample_rate=16000, window_size=1024, hop_size=320,
        mel_bins=64, fmin=50, fmax=8000, classes_num=527
    )
    eval_model = CNN14BinaryClassifier(eval_backbone).to(device)
    eval_model.load_state_dict(payload["model_state_dict"])

    # Evaluate without optimizer updates.
    eval_model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    with torch.no_grad():
        for waveforms, labels in tqdm(loader, leave=False):
            waveforms = waveforms.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = eval_model(waveforms)
            loss = criterion(logits, labels)
            total_loss += loss.item() * waveforms.size(0)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(logits.argmax(dim=1).cpu().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(total_loss / len(loader.dataset))
    print(f"\n{label} checkpoint — epoch {payload['epoch']}")
    print(f"Test loss:   {metrics['loss']:.4f}")
    print(f"Accuracy:    {metrics['accuracy']:.4f}")
    print(f"Sensitivity: {metrics['sensitivity']:.4f}")
    print(f"Specificity: {metrics['specificity']:.4f}")
    print(f"HS-score:    {metrics['HS_score']:.4f}")
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(np.array(metrics["confusion_matrix"]))

    return {
        "checkpoint": str(checkpoint_path),
        "epoch": int(payload["epoch"]),
        "test_loss": metrics["loss"],
        "test_accuracy": metrics["accuracy"],
        "test_sensitivity": metrics["sensitivity"],
        "test_specificity": metrics["specificity"],
        "test_HS_score": metrics["HS_score"],
        "confusion_matrix": metrics["confusion_matrix"],
    }

best_test = evaluate_checkpoint(BEST_CHECKPOINT, test_loader, "BEST validation-HS")
last_test = evaluate_checkpoint(LAST_CHECKPOINT, test_loader, "LAST epoch")


BEST validation-HS checkpoint — epoch 14
Test loss:   0.6828
Accuracy:    0.5711
Sensitivity: 0.5718
Specificity: 0.5706
HS-score:    0.5712
Confusion matrix [[TN, FP], [FN, TP]]:
[[901 678]
 [504 673]]



LAST epoch checkpoint — epoch 20
Test loss:   0.6892
Accuracy:    0.5548
Sensitivity: 0.6406
Specificity: 0.4908
HS-score:    0.5558
Confusion matrix [[TN, FP], [FN, TP]]:
[[775 804]
 [423 754]]


In [21]:
# 10. Save test results and a compact comparison table
results = {
    "checkpoint_selection": "highest validation HS-score",
    "best_epoch": int(best_epoch),
    "best_validation_HS_score": float(best_val_hs),
    "best_checkpoint_test": best_test,
    "last_checkpoint_test": last_test,
    "split_counts": {
        "train_segments": int(len(train_df)),
        "validation_segments": int(len(val_df)),
        "official_test_segments": int(len(test_df)),
        "train_patients": int(train_df.patient_id.nunique()),
        "validation_patients": int(val_df.patient_id.nunique()),
        "official_test_patients": int(test_df.patient_id.nunique()),
    },
}
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

display(pd.DataFrame([
    {"checkpoint": "best_validation_HS", **{k: best_test[k] for k in [
        "epoch", "test_loss", "test_accuracy", "test_sensitivity",
        "test_specificity", "test_HS_score"
    ]}},
    {"checkpoint": "last_epoch", **{k: last_test[k] for k in [
        "epoch", "test_loss", "test_accuracy", "test_sensitivity",
        "test_specificity", "test_HS_score"
    ]}},
]))
print("Saved results:", RESULTS_PATH.resolve())

,checkpoint,epoch,test_loss,test_accuracy,test_sensitivity,test_specificity,test_HS_score
0,best_validation_HS,14,0.682782,0.571118,0.571793,0.570614,0.571203
1,last_epoch,20,0.689190,0.554790,0.640612,0.490817,0.555798


Saved results: D:\.JKU CS courses\Master Thesis\models\cnn14_end_to_end_binary\test_results.json
